# 03LIC_1071 PVLO Alarm Analysis from Events Data

Analyzing alarm behavior for `03LIC_1071` PVLO alarms extracted from the events dataset.

**Goal**: Understand alarm patterns — are these mostly chattering alarms (rapid on/off) or sustained alarms?

In [23]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Load events data
events_df = pd.read_csv("../DATA/df_df_events_1071_export.csv", low_memory=False)
events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)

# Deduplicate: for rows with same (VT_Start, Source, ConditionName), keep the row with the most non-null values
dedup_cols = ['VT_Start', 'Source', 'ConditionName']
pre_dedup = len(events_df)
events_df['_non_null_count'] = events_df.notna().sum(axis=1)
events_df = events_df.sort_values(dedup_cols + ['_non_null_count'], ascending=[True, True, True, False])
events_df = events_df.drop_duplicates(subset=dedup_cols, keep='first')
events_df = events_df.drop(columns='_non_null_count').reset_index(drop=True)
print(f"Deduplication: {pre_dedup} → {len(events_df)} rows (removed {pre_dedup - len(events_df)} duplicates)")

events_df = events_df[events_df['VT_Start'] >= pd.to_datetime("2025-01-01")]

# Filter: 03LIC_1071 PVLO, Category=1 (actual alarm events, not ACK/SHELVE)
pvlo = events_df[
    (events_df['Source'] == '03LIC_1071') &
    (events_df['ConditionName'] == 'PVLO') &
    (events_df['Category'] == 1)
].copy()

print(f"Total PVLO Category=1 events: {len(pvlo)}")
print(f"  Alarm starts (Action is NaN/blank): {pvlo['Action'].isna().sum().sum()}")
print(f"  Alarm ends (Action = OK): {(pvlo['Action'] == 'OK').sum()}")
print(f"  Date range: {pvlo['VT_Start'].min()} to {pvlo['VT_Start'].max()}")
print(f"\nAction breakdown:")
print(pvlo['Action'].value_counts(dropna=False))

Deduplication: 1947510 → 1799233 rows (removed 148277 duplicates)
Total PVLO Category=1 events: 1312
  Alarm starts (Action is NaN/blank): 656
  Alarm ends (Action = OK): 656
  Date range: 2025-01-03 06:47:57.192900 to 2025-06-22 17:14:56.204800

Action breakdown:
Action
NaN    656
OK     656
Name: count, dtype: int64


In [24]:
# Extract alarm episodes by walking through events in order
# Rule: Start = Action is NaN, End = Action is OK
# If we see multiple starts in a row, alarm is still ongoing (PV bouncing near threshold)
# Take the FIRST start followed by the NEXT OK as one episode

episodes_list = []
current_start = None
current_start_value = None

for _, row in pvlo.iterrows():
    is_start = pd.isna(row['Action']) or row['Action'] == ''
    is_end = row['Action'] == 'OK'
    
    if is_start and current_start is None:
        current_start = row['VT_Start']
        current_start_value = row['Value']
    elif is_start and current_start is not None:
        pass  # still in alarm
    elif is_end and current_start is not None:
        episodes_list.append({
            'alarm_start': current_start,
            'alarm_end': row['VT_Start'],
            'start_value': current_start_value,
            'end_value': row['Value'],
        })
        current_start = None
        current_start_value = None

episodes = pd.DataFrame(episodes_list)
episodes['episode_num'] = range(1, len(episodes) + 1)
episodes['duration_minutes'] = (episodes['alarm_end'] - episodes['alarm_start']).dt.total_seconds() / 60
episodes['gap_to_next_minutes'] = (
    episodes['alarm_start'].shift(-1) - episodes['alarm_end']
).dt.total_seconds() / 60

print(f"Total alarm episodes: {len(episodes)}")
print(f"Orphan starts (no matching end): {pvlo['Action'].isna().sum() - len(episodes)}")
print(f"\nAlarm DURATION (minutes):")
print(episodes['duration_minutes'].describe().to_string())
print(f"\nGAP to next alarm (minutes):")
print(episodes['gap_to_next_minutes'].dropna().describe().to_string())

Total alarm episodes: 656
Orphan starts (no matching end): 0

Alarm DURATION (minutes):
count    656.000000
mean       3.399565
std       18.258505
min        0.008822
25%        0.195243
50%        0.749912
75%        2.037828
max      330.592688

GAP to next alarm (minutes):
count      655.000000
mean       371.292929
std       3703.736404
min          0.026467
25%          0.321052
50%          1.734333
75%          7.766203
max      65969.333103


In [25]:
# Classify alarms by duration
def classify_alarm(duration_min):
    if duration_min <= 1:
        return '≤1 min (fleeting)'
    elif duration_min <= 5:
        return '1-5 min (chattering)'
    elif duration_min <= 30:
        return '5-30 min (short)'
    elif duration_min <= 60:
        return '30-60 min (medium)'
    else:
        return '>60 min (sustained)'

episodes['alarm_class'] = episodes['duration_minutes'].apply(classify_alarm)

class_order = ['≤1 min (fleeting)', '1-5 min (chattering)', '5-30 min (short)', 
               '30-60 min (medium)', '>60 min (sustained)']
class_counts = episodes['alarm_class'].value_counts().reindex(class_order).fillna(0).astype(int)
class_pcts = (class_counts / len(episodes) * 100).round(1)

print("Alarm Duration Classification:")
print("=" * 50)
for cls in class_order:
    print(f"  {cls:25s}: {class_counts[cls]:5d} ({class_pcts[cls]:5.1f}%)")
print(f"  {'TOTAL':25s}: {len(episodes):5d}")

Alarm Duration Classification:
  ≤1 min (fleeting)        :   364 ( 55.5%)
  1-5 min (chattering)     :   237 ( 36.1%)
  5-30 min (short)         :    46 (  7.0%)
  30-60 min (medium)       :     2 (  0.3%)
  >60 min (sustained)      :     7 (  1.1%)
  TOTAL                    :   656


In [26]:
# Distribution of alarm DURATIONS
fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Alarm Duration Distribution (all episodes)',
    'Alarm Duration Distribution (zoomed ≤60 min)'
], vertical_spacing=0.12)

fig.add_trace(go.Histogram(x=episodes['duration_minutes'], nbinsx=100, 
                            marker_color='indianred', name='All'), row=1, col=1)
fig.add_trace(go.Histogram(x=episodes[episodes['duration_minutes'] <= 60]['duration_minutes'], 
                            nbinsx=60, marker_color='steelblue', name='≤60 min'), row=2, col=1)

fig.update_xaxes(title_text='Duration (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Duration (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, title_text='How long do alarms last?')
fig.show()

In [27]:
# Distribution of GAP between consecutive alarms
gaps = episodes['gap_to_next_minutes'].dropna()

# Classify gaps
def classify_gap(gap_min):
    if gap_min <= 5:
        return '≤5 min (rapid re-alarm)'
    elif gap_min <= 30:
        return '5-30 min'
    elif gap_min <= 60:
        return '30-60 min'
    elif gap_min <= 360:
        return '1-6 hours'
    elif gap_min <= 1440:
        return '6-24 hours'
    else:
        return '>24 hours'

gap_classes = gaps.apply(classify_gap)
gap_order = ['≤5 min (rapid re-alarm)', '5-30 min', '30-60 min', '1-6 hours', '6-24 hours', '>24 hours']
gap_counts = gap_classes.value_counts().reindex(gap_order).fillna(0).astype(int)
gap_pcts = (gap_counts / len(gaps) * 100).round(1)

print("Gap Between Consecutive Alarms:")
print("=" * 50)
for cls in gap_order:
    print(f"  {cls:25s}: {gap_counts[cls]:5d} ({gap_pcts[cls]:5.1f}%)")

fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Gap to Next Alarm (all)',
    'Gap to Next Alarm (zoomed ≤120 min)'
], vertical_spacing=0.12)

fig.add_trace(go.Histogram(x=gaps, nbinsx=100, marker_color='darkorange', name='All'), row=1, col=1)
fig.add_trace(go.Histogram(x=gaps[gaps <= 120], nbinsx=60, marker_color='teal', name='≤120 min'), row=2, col=1)

fig.update_xaxes(title_text='Gap to next alarm (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Gap to next alarm (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, title_text='How quickly does the alarm come back?')
fig.show()

Gap Between Consecutive Alarms:
  ≤5 min (rapid re-alarm)  :   461 ( 70.4%)
  5-30 min                 :   100 ( 15.3%)
  30-60 min                :    26 (  4.0%)
  1-6 hours                :    24 (  3.7%)
  6-24 hours               :    22 (  3.4%)
  >24 hours                :    22 (  3.4%)


In [28]:
# Filter to 2025 and build clusters (30 min gap threshold)
CLUSTER_GAP_THRESHOLD = 30  # minutes

episodes_2025 = episodes[episodes['alarm_start'].dt.year == 2025].copy().reset_index(drop=True)
print(f"2025 alarm episodes: {len(episodes_2025)} (out of {len(episodes)} total)")

# Recompute gaps for 2025
episodes_2025['gap_to_next_minutes'] = (
    episodes_2025['alarm_start'].shift(-1) - episodes_2025['alarm_end']
).dt.total_seconds() / 60

# Assign cluster_id: consecutive alarms with gap ≤ threshold belong to the same cluster
cluster_id = 0
cluster_ids = [0]
for g in episodes_2025['gap_to_next_minutes'].iloc[:-1]:
    if pd.notna(g) and g <= CLUSTER_GAP_THRESHOLD:
        cluster_ids.append(cluster_id)
    else:
        cluster_id += 1
        cluster_ids.append(cluster_id)

episodes_2025['cluster_id'] = cluster_ids

# Compute cluster-level stats
clusters = episodes_2025.groupby('cluster_id').agg(
    cluster_start=('alarm_start', 'min'),
    cluster_end=('alarm_end', 'max'),
    n_alarms=('episode_num', 'count')
).sort_values('cluster_start')

clusters['total_duration_min'] = (clusters['cluster_end'] - clusters['cluster_start']).dt.total_seconds() / 60
clusters['gap_to_next_cluster_min'] = (
    clusters['cluster_start'].shift(-1) - clusters['cluster_end']
).dt.total_seconds() / 60

# Classify cluster types
def classify_cluster(row):
    if row['n_alarms'] == 1 and row['total_duration_min'] <= 5:
        return 'Isolated brief alarm'
    elif row['n_alarms'] <= 3 and row['total_duration_min'] <= 30:
        return 'Small cluster'
    elif row['total_duration_min'] <= 120:
        return 'Medium situation (<2h)'
    else:
        return 'Extended situation (>2h)'

clusters['cluster_type'] = clusters.apply(classify_cluster, axis=1)

# Summary
print(f"\n{len(episodes_2025)} raw alarms → {len(clusters)} independent clusters (gap threshold: {CLUSTER_GAP_THRESHOLD} min)")
print(f"  Single-alarm: {(clusters['n_alarms'] == 1).sum()}")
print(f"  Multi-alarm: {(clusters['n_alarms'] > 1).sum()}")
print(f"\nCluster sizes (# alarms per cluster):")
print(clusters['n_alarms'].describe().to_string())
print(f"\nGap between clusters (minutes):")
print(clusters['gap_to_next_cluster_min'].dropna().describe().to_string())
print(f"\nCluster types:")
for ctype in ['Isolated brief alarm', 'Small cluster', 'Medium situation (<2h)', 'Extended situation (>2h)']:
    count = (clusters['cluster_type'] == ctype).sum()
    pct = count / len(clusters) * 100
    avg_alarms = clusters[clusters['cluster_type'] == ctype]['n_alarms'].mean()
    avg_dur = clusters[clusters['cluster_type'] == ctype]['total_duration_min'].mean()
    print(f"  {ctype:30s}: {count:4d} ({pct:5.1f}%) | avg {avg_alarms:.1f} alarms, avg {avg_dur:.0f} min")

2025 alarm episodes: 656 (out of 656 total)

656 raw alarms → 95 independent clusters (gap threshold: 30 min)
  Single-alarm: 39
  Multi-alarm: 56

Cluster sizes (# alarms per cluster):
count    95.000000
mean      6.905263
std      15.399001
min       1.000000
25%       1.000000
50%       2.000000
75%       4.500000
max      94.000000

Gap between clusters (minutes):
count       94.000000
mean      2568.620960
std       9527.230189
min         31.214452
25%         55.339845
50%        281.832683
75%       1238.180252
max      65969.333103

Cluster types:
  Isolated brief alarm          :   38 ( 40.0%) | avg 1.0 alarms, avg 1 min
  Small cluster                 :   26 ( 27.4%) | avg 2.2 alarms, avg 14 min
  Medium situation (<2h)        :   21 ( 22.1%) | avg 14.0 alarms, avg 60 min
  Extended situation (>2h)      :   10 ( 10.5%) | avg 26.7 alarms, avg 232 min


In [29]:
# Distribution of gaps between clusters
cluster_gaps = clusters['gap_to_next_cluster_min'].dropna()

def classify_cluster_gap(gap_min):
    if gap_min <= 60:
        return '≤1 hour'
    elif gap_min <= 360:
        return '1-6 hours'
    elif gap_min <= 1440:
        return '6-24 hours'
    elif gap_min <= 4320:
        return '1-3 days'
    elif gap_min <= 10080:
        return '3-7 days'
    else:
        return '>7 days'

gap_cls = cluster_gaps.apply(classify_cluster_gap)
gap_order = ['≤1 hour', '1-6 hours', '6-24 hours', '1-3 days', '3-7 days', '>7 days']
gap_counts = gap_cls.value_counts().reindex(gap_order).fillna(0).astype(int)
gap_pcts = (gap_counts / len(cluster_gaps) * 100).round(1)

print(f"Gap Between Alarm Clusters ({CLUSTER_GAP_THRESHOLD}-min threshold, 2025):")
print("=" * 60)
for cls in gap_order:
    print(f"  {cls:20s}: {gap_counts[cls]:5d} ({gap_pcts[cls]:5.1f}%)")

fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Gap Between Clusters — All (2025)',
    'Gap Between Clusters — Zoomed ≤1440 min / 24h (2025)'
], vertical_spacing=0.15)

fig.add_trace(go.Histogram(x=cluster_gaps, nbinsx=80, marker_color='mediumpurple'), row=1, col=1)
fig.add_trace(go.Histogram(x=cluster_gaps[cluster_gaps <= 1440], nbinsx=60, marker_color='mediumseagreen'), row=2, col=1)

fig.update_xaxes(title_text='Gap (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Gap (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, 
                  title_text=f'Gap between alarm clusters ({CLUSTER_GAP_THRESHOLD}-min threshold, 2025)')
fig.show()

Gap Between Alarm Clusters (30-min threshold, 2025):
  ≤1 hour             :    26 ( 27.7%)
  1-6 hours           :    24 ( 25.5%)
  6-24 hours          :    22 ( 23.4%)
  1-3 days            :    14 ( 14.9%)
  3-7 days            :     5 (  5.3%)
  >7 days             :     3 (  3.2%)


In [30]:
# Try multiple gap thresholds to find truly independent alarm situations
for threshold in [5, 15, 30, 60, 120]:
    cid = 0
    cids = [0]
    for g in episodes_2025['gap_to_next_minutes'].iloc[:-1]:
        if pd.notna(g) and g <= threshold:
            cids.append(cid)
        else:
            cid += 1
            cids.append(cid)
    n_clusters = cid + 1
    # Compute inter-cluster gaps for this threshold
    episodes_2025[f'_cid_{threshold}'] = cids
    cd = episodes_2025.groupby(f'_cid_{threshold}').agg(
        start=('alarm_start', 'min'), end=('alarm_end', 'max'),
        n_alarms=('episode_num', 'count')
    ).sort_values('start')
    cd['gap_next'] = (cd['start'].shift(-1) - cd['end']).dt.total_seconds() / 60
    median_gap = cd['gap_next'].dropna().median()
    pct_under_30 = (cd['gap_next'].dropna() <= 30).mean() * 100
    
    print(f"Threshold {threshold:>3d} min → {n_clusters:>4d} clusters | "
          f"median inter-cluster gap: {median_gap:>8.1f} min | "
          f"% clusters with gap ≤30 min: {pct_under_30:.1f}%")

Threshold   5 min →  195 clusters | median inter-cluster gap:     27.5 min | % clusters with gap ≤30 min: 51.5%
Threshold  15 min →  123 clusters | median inter-cluster gap:     82.2 min | % clusters with gap ≤30 min: 23.0%
Threshold  30 min →   95 clusters | median inter-cluster gap:    281.8 min | % clusters with gap ≤30 min: 0.0%
Threshold  60 min →   69 clusters | median inter-cluster gap:    845.4 min | % clusters with gap ≤30 min: 0.0%
Threshold 120 min →   56 clusters | median inter-cluster gap:   1125.4 min | % clusters with gap ≤30 min: 0.0%


In [31]:
# Build output dataframe: one row per raw alarm, with cluster info
output = episodes_2025[['episode_num', 'alarm_start', 'alarm_end', 'duration_minutes', 
                         'gap_to_next_minutes', 'start_value', 'end_value', 'cluster_id']].copy()

# Add cluster-level info
output = output.merge(
    clusters[['n_alarms', 'cluster_start', 'cluster_end', 'total_duration_min', 'gap_to_next_cluster_min', 'cluster_type']],
    left_on='cluster_id', right_index=True
)
output = output.rename(columns={
    'n_alarms': 'cluster_total_alarms',
    'cluster_start': 'cluster_start_time',
    'cluster_end': 'cluster_end_time',
    'total_duration_min': 'cluster_total_duration_min'
})

# Renumber clusters from 1
cluster_id_map = {old: new for new, old in enumerate(sorted(output['cluster_id'].unique()), 1)}
output['cluster_id'] = output['cluster_id'].map(cluster_id_map)
output = output.sort_values('alarm_start').reset_index(drop=True)

# --- Extract operator control actions (CHANGE events) per cluster ---
# Window: [cluster_start - 30 min, cluster_end + 30 min]
WINDOW_MARGIN = pd.Timedelta(minutes=30)

change_events = events_df[
    events_df['ConditionName'] == 'CHANGE'
].copy()

# Build cluster lookup with renumbered IDs
cluster_lookup = clusters.copy()
cluster_lookup['cluster_id_new'] = cluster_lookup.index.map(cluster_id_map)
cluster_lookup = cluster_lookup.sort_values('cluster_start').reset_index(drop=True)

actions_list = []
for _, cl in cluster_lookup.iterrows():
    window_start = cl['cluster_start'] - WINDOW_MARGIN
    window_end = cl['cluster_end'] + WINDOW_MARGIN
    
    mask = (change_events['VT_Start'] >= window_start) & (change_events['VT_Start'] <= window_end)
    cluster_actions = change_events[mask].copy()
    cluster_actions['cluster_id'] = cl['cluster_id_new']
    cluster_actions['cluster_start'] = cl['cluster_start']
    cluster_actions['cluster_end'] = cl['cluster_end']
    actions_list.append(cluster_actions)

control_actions = pd.concat(actions_list, ignore_index=True)
control_actions = control_actions.sort_values(['cluster_id', 'VT_Start']).reset_index(drop=True)

# Classify action timing relative to cluster duration
control_actions['action_timing'] = np.where(
    control_actions['VT_Start'] < control_actions['cluster_start'], 'before',
    np.where(control_actions['VT_Start'] > control_actions['cluster_end'], 'after', 'during')
)

# Classify action direction based on Value vs PrevValue
control_actions['Value'] = pd.to_numeric(control_actions['Value'], errors='coerce')
control_actions['PrevValue'] = pd.to_numeric(control_actions['PrevValue'], errors='coerce')
control_actions['action_direction'] = np.where(
    control_actions['Value'] > control_actions['PrevValue'], 'increase',
    np.where(control_actions['Value'] < control_actions['PrevValue'], 'decrease', 'no_change')
)

print(f"Control actions extracted: {len(control_actions)} CHANGE events across {control_actions['cluster_id'].nunique()} clusters")
print(f"Clusters with no actions: {output['cluster_id'].nunique() - control_actions['cluster_id'].nunique()}")
print(f"\nAction timing breakdown:")
print(control_actions['action_timing'].value_counts().to_string())
print(f"\nTop 10 most acted-on tags:")
print(control_actions['Source'].value_counts().head(10).to_string())

# Save both sheets to Excel
output_path = '../DATA/1071_pvlo_alarms_2025_clustered.xlsx'
actions_save_cols = ['cluster_id', 'cluster_start', 'cluster_end', 'action_timing', 'action_direction', 'Source', 'Description', 'VT_Start', 'PrevValue', 'Value']
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    output.to_excel(writer, sheet_name='alarm_clusters', index=False)
    control_actions[actions_save_cols].to_excel(writer, sheet_name='control_actions', index=False)

print(f"\nSaved to {output_path}")
print(f"  Sheet 'alarm_clusters': {len(output)} alarms across {output['cluster_id'].nunique()} clusters")
print(f"  Sheet 'control_actions': {len(control_actions)} actions")
output.head(10)

Control actions extracted: 4415 CHANGE events across 84 clusters
Clusters with no actions: 11

Action timing breakdown:
action_timing
during    2431
before    1067
after      917

Top 10 most acted-on tags:
Source
03FIC_3435     649
03PIC_1013     616
03LIC_1071     480
03HIC_1141     451
03HIC_1151     432
03PIC_3131     252
03HIC_3100     222
03LIC_1034     167
03HIC_3132     160
03FIC_3435A    116

Saved to ../DATA/1071_pvlo_alarms_2025_clustered.xlsx
  Sheet 'alarm_clusters': 656 alarms across 95 clusters
  Sheet 'control_actions': 4415 actions


,episode_num,alarm_start,alarm_end,duration_minutes,gap_to_next_minutes,start_value,end_value,cluster_id,cluster_total_alarms,cluster_start_time,cluster_end_time,cluster_total_duration_min,gap_to_next_cluster_min,cluster_type
0,1,2025-01-03 06:47:57.192900,2025-01-03 06:53:45.596200,5.806722,10.075365,28.651,31.953,1,5,2025-01-03 06:47:57.192900,2025-01-03 07:55:14.178100,67.283087,37.385520,Medium situation (<2h)
1,2,2025-01-03 07:03:50.118100,2025-01-03 07:10:05.131200,6.250218,9.038250,28.675,31.788,1,5,2025-01-03 06:47:57.192900,2025-01-03 07:55:14.178100,67.283087,37.385520,Medium situation (<2h)
2,3,2025-01-03 07:19:07.426200,2025-01-03 07:23:45.424800,4.633310,10.119357,28.637,31.861,1,5,2025-01-03 06:47:57.192900,2025-01-03 07:55:14.178100,67.283087,37.385520,Medium situation (<2h)
3,4,2025-01-03 07:33:52.586200,2025-01-03 07:38:28.130800,4.592410,10.023243,28.643,31.752,1,5,2025-01-03 06:47:57.192900,2025-01-03 07:55:14.178100,67.283087,37.385520,Medium situation (<2h)
4,5,2025-01-03 07:48:29.525400,2025-01-03 07:55:14.178100,6.744212,37.385520,28.733,31.794,1,5,2025-01-03 06:47:57.192900,2025-01-03 07:55:14.178100,67.283087,37.385520,Medium situation (<2h)
5,6,2025-01-03 08:32:37.309300,2025-01-03 08:35:03.420500,2.435187,17.864662,28.740,31.763,2,2,2025-01-03 08:32:37.309300,2025-01-03 08:58:30.512700,25.886723,2778.381567,Small cluster
6,7,2025-01-03 08:52:55.300200,2025-01-03 08:58:30.512700,5.586875,2778.381567,28.631,31.923,2,2,2025-01-03 08:32:37.309300,2025-01-03 08:58:30.512700,25.886723,2778.381567,Small cluster
7,8,2025-01-05 07:16:53.406700,2025-01-05 07:20:53.406900,4.000003,8.383470,28.668,31.902,3,5,2025-01-05 07:16:53.406700,2025-01-05 08:31:43.446600,74.833998,1123.593742,Medium situation (<2h)
8,9,2025-01-05 07:29:16.415100,2025-01-05 07:38:42.523400,9.435138,13.500192,28.640,31.797,3,5,2025-01-05 07:16:53.406700,2025-01-05 08:31:43.446600,74.833998,1123.593742,Medium situation (<2h)
9,10,2025-01-05 07:52:12.534900,2025-01-05 07:55:13.156000,3.010352,7.417747,28.706,31.944,3,5,2025-01-05 07:16:53.406700,2025-01-05 08:31:43.446600,74.833998,1123.593742,Medium situation (<2h)


In [33]:
output['gap_to_next_cluster_min'].describe()

count      655.000000
mean      2256.905466
std       7406.137314
min         31.214452
25%        106.718575
50%        504.481000
75%       1447.191485
max      65969.333103
Name: gap_to_next_cluster_min, dtype: float64

In [32]:
# Control action stats for 03LIC_1071, 03LIC_1016, 03PIC_1013
target_tags = ['03LIC_1071', '03LIC_1016', '03PIC_1013']
control_actions['change_magnitude'] = (control_actions['Value'] - control_actions['PrevValue']).abs()

for tag in target_tags:
    tag_actions = control_actions[control_actions['Source'] == tag]
    print(f"\n{'='*60}")
    print(f"Tag: {tag} — {len(tag_actions)} total actions")
    print(f"{'='*60}")
    
    dir_counts = tag_actions['action_direction'].value_counts()
    for d in ['increase', 'decrease', 'no_change']:
        subset = tag_actions[tag_actions['action_direction'] == d]
        n = len(subset)
        if n == 0:
            print(f"\n  {d}: 0 actions")
            continue
        mag = subset['change_magnitude']
        print(f"\n  {d}: {n} actions")
        print(f"    magnitude — mean: {mag.mean():.2f}, median: {mag.median():.2f}, "
              f"min: {mag.min():.2f}, max: {mag.max():.2f}, std: {mag.std():.2f}")


Tag: 03LIC_1071 — 480 total actions

  increase: 124 actions
    magnitude — mean: 2.38, median: 2.00, min: 0.91, max: 18.86, std: 2.30

  decrease: 100 actions
    magnitude — mean: 2.96, median: 2.00, min: 0.87, max: 46.44, std: 6.12

  no_change: 256 actions
    magnitude — mean: nan, median: nan, min: nan, max: nan, std: nan

Tag: 03LIC_1016 — 78 total actions

  increase: 20 actions
    magnitude — mean: 2.04, median: 2.00, min: 0.81, max: 5.00, std: 0.77

  decrease: 10 actions
    magnitude — mean: 4.90, median: 3.50, min: 1.00, max: 16.00, std: 4.51

  no_change: 48 actions
    magnitude — mean: nan, median: nan, min: nan, max: nan, std: nan

Tag: 03PIC_1013 — 616 total actions

  increase: 95 actions
    magnitude — mean: 2.26, median: 2.00, min: 1.00, max: 35.00, std: 3.41

  decrease: 193 actions
    magnitude — mean: 1.12, median: 1.00, min: 0.10, max: 14.00, std: 1.27

  no_change: 328 actions
    magnitude — mean: 0.00, median: 0.00, min: 0.00, max: 0.00, std: 0.00
